# O que faz um vídeo do CalangoLive ter mais views?

Análise exploratória e estatística dos **2.960 vídeos** do canal [@CalangoLive](https://www.youtube.com/@CalangoLive/videos), coletados da página pública do canal em 23/11/2026 e modelados com dbt no BigQuery.

**Perguntas:**
1. Como o canal mudou ao longo do tempo?
2. Quais jogos trazem mais audiência e quais só ocupam a agenda?
3. Quanto da audiência sobra do episódio 1 até o episódio 10 de uma série?
4. Duração, collab, desafio e estilo de título se associam a mais views?

**Limitações (leia antes de concluir qualquer coisa):**
- Views são **totais acumulados**: vídeos antigos tiveram mais tempo para somar views.
- Views vêm arredondadas pelo YouTube ("3,9 mil"), e a data de publicação dos vídeos com mais de um ano é **estimada**.
- É um estudo **observacional**: associação não é causa. Aqui não há grupo de controle, diferente de um teste A/B.

In [ ]:
# ==== CONFIGURAÇÃO ====
DATA_SOURCE = 'csv'   # 'csv' (arquivos do repositório no GitHub) ou 'bigquery' (tabelas criadas pelo dbt)
GITHUB_RAW = 'https://raw.githubusercontent.com/jmartellet/gameplay-studies/main/calango-analytics/data'
PROJECT_ID = 'seu-projeto-gcp'      # usado só se DATA_SOURCE = 'bigquery'
DBT_DATASET = 'dbt_seu_nome'        # dataset onde o dbt criou as tabelas

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf
pd.set_option('display.float_format', '{:,.2f}'.format)
plt.rcParams.update({'figure.figsize': (8, 4), 'axes.grid': True, 'grid.alpha': .3})

def load(table):
    if DATA_SOURCE == 'bigquery':
        from google.colab import auth
        auth.authenticate_user()
        from google.cloud import bigquery
        return bigquery.Client(project=PROJECT_ID).query(f'select * from `{PROJECT_ID}.{DBT_DATASET}.{table}`').to_dataframe()
    return pd.read_csv(f'{GITHUB_RAW}/{table}.csv')

videos = load('fct_calango_videos')
games  = load('fct_calango_games')
decay  = load('fct_calango_series_decay')
curve  = load('fct_calango_decay_curve').sort_values('episode_number')
videos['approx_published_date'] = pd.to_datetime(videos['approx_published_date'])
print(f'{len(videos):,} vídeos | {videos.views.sum()/1e6:,.1f} mi de views | {videos.duration_sec.sum()/3600:,.0f} horas de conteúdo')

## 1. Visão geral

Views de YouTube têm distribuição muito assimétrica: poucos vídeos concentram a maior parte. Por isso a **mediana** é mais honesta que a média, e os gráficos usam escala log.

In [ ]:
print(videos.views.describe(percentiles=[.1, .25, .5, .75, .9, .99]).to_string())
top10_share = videos.nlargest(int(len(videos) * .01), 'views').views.sum() / videos.views.sum()
print(f'\nOs 1% vídeos mais vistos concentram {top10_share:.0%} de todas as views')
plt.hist(np.log10(videos.views), bins=50, edgecolor='white')
plt.xticks([3, 4, 5, 6], ['1 mil', '10 mil', '100 mil', '1 mi']); plt.title('Distribuição de views por vídeo (escala log)'); plt.show()
videos.nlargest(10, 'views')[['title', 'views', 'published_year', 'game']]

## 2. Como o canal mudou ao longo do tempo

In [ ]:
yearly = videos.groupby('published_year').agg(
    videos=('video_id', 'size'), horas=('duration_sec', lambda s: s.sum() / 3600),
    mediana_views=('views', 'median'), pct_caixa_alta=('title_case', lambda s: (s == 'CAIXA ALTA').mean()),
    jogos_distintos=('game', 'nunique'))
display(yearly)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
yearly.videos.plot.bar(ax=ax[0], title='Vídeos publicados por ano')
yearly.mediana_views.plot(ax=ax[1], marker='o', title='Mediana de views por ano de publicação')
plt.tight_layout(); plt.show()

**Para escrever:** em que ano o estilo de título mudou de "CALANGO JOGANDO X #N" para "frase | jogo #N"? O que aconteceu com o volume e a mediana de views depois disso? Lembre do viés de views acumuladas: anos antigos tendem a parecer melhores do que foram.

## 3. Jogos: audiência vs. espaço na agenda

In [ ]:
g = games[games.videos >= 5].copy()
g['pct_videos'] = g.videos / g.videos.sum()
g['pct_views'] = g.total_views / g.total_views.sum()
print('Mais views no total (jogos com 5+ vídeos):')
display(g.nlargest(10, 'total_views')[['game', 'genre', 'videos', 'total_views', 'median_views']])
print('Mais "eficientes": maior mediana de views por vídeo:')
display(g.nlargest(10, 'median_views')[['game', 'genre', 'videos', 'median_views']])
print('Muito espaço na agenda, pouca audiência por vídeo:')
display(g[g.videos >= 20].nsmallest(8, 'median_views')[['game', 'videos', 'median_views', 'hours_of_content']])

In [ ]:
plt.figure(figsize=(9, 6))
plt.scatter(g.videos, g.median_views, s=np.sqrt(g.total_views) / 15, alpha=.5)
for _, r in g[(g.videos >= 40) | (g.median_views >= 60000)].iterrows():
    plt.annotate(r.game, (r.videos, r.median_views), fontsize=8)
plt.xscale('log'); plt.yscale('log'); plt.xlabel('Vídeos do jogo (log)'); plt.ylabel('Mediana de views (log)')
plt.title('Cada bolha é um jogo (tamanho = views totais)'); plt.show()
genre = videos[videos.content_type == 'Gameplay'].groupby('genre').agg(videos=('video_id', 'size'), mediana_views=('views', 'median')).sort_values('mediana_views', ascending=False)
genre

## 4. Séries: quanto da audiência sobra a cada episódio?

Tabela `fct_calango_series_decay` (window functions: `FIRST_VALUE` e `LAG`). A retenção é `views do episódio N / views do episódio 1`.

In [ ]:
plt.plot(curve.episode_number, curve.median_retention_vs_episode_1 * 100, marker='o', label='mediana')
plt.fill_between(curve.episode_number, curve.p25_retention * 100, curve.p75_retention * 100, alpha=.2, label='p25–p75')
plt.ylabel('% das views do episódio 1'); plt.xlabel('Episódio'); plt.title('Curva de queda de audiência em séries'); plt.legend(); plt.show()
curve[['episode_number', 'series_with_this_episode', 'median_retention_vs_episode_1']].head(10)

In [ ]:
# séries que seguraram melhor a audiência no episódio 5
ep5 = decay[decay.episode_number == 5][['series_name', 'game', 'episode_1_views', 'views', 'retention_vs_episode_1']]
print('Melhor retenção no episódio 5:'); display(ep5.nlargest(8, 'retention_vs_episode_1'))
print('Pior retenção no episódio 5:');   display(ep5.nsmallest(8, 'retention_vs_episode_1'))

## 5. Testes de hipótese: o que se associa a mais views?

Views não são normais, então usamos **Mann-Whitney U**, teste não paramétrico que compara distribuições sem supor normalidade, e reportamos a razão das medianas.

**Armadilha clássica:** comparar títulos em CAIXA ALTA com títulos minúsculos mistura o efeito do título com o efeito da **época** (todos os títulos antigos são em caixa alta). Por isso a comparação é refeita **dentro do mesmo ano**.

In [ ]:
def compare(df, flag, label):
    a, b = df.loc[df[flag], 'views'], df.loc[~df[flag], 'views']
    if len(a) < 10 or len(b) < 10:
        return None
    u, p = stats.mannwhitneyu(a, b, alternative='two-sided')
    return {'comparação': label, 'n com': len(a), 'n sem': len(b), 'mediana com': a.median(),
            'mediana sem': b.median(), 'razão': a.median() / b.median(), 'p-valor': p}

gp = videos[videos.content_type == 'Gameplay'].copy()
gp['is_episode_1'] = gp.episode_number == 1
gp['is_caps'] = gp.title_case == 'CAIXA ALTA'
pd.DataFrame([r for r in [
    compare(gp, 'is_episode_1', 'Episódio 1 vs. demais'),
    compare(gp, 'is_collab', 'Collab vs. solo'),
    compare(gp, 'is_challenge', 'Desafio vs. normal'),
    compare(gp, 'title_has_question', 'Título com ? vs. sem'),
    compare(gp, 'title_has_emoji', 'Título com emoji vs. sem'),
    compare(gp, 'is_caps', 'CAIXA ALTA vs. não (ingênuo!)'),
] if r])

In [ ]:
# mesma comparação de CAIXA ALTA, agora estratificada por ano
rows = []
for year, df in gp.groupby('published_year'):
    r = compare(df, 'is_caps', f'CAIXA ALTA vs. não em {year}')
    if r: rows.append(r)
pd.DataFrame(rows) if rows else 'Nenhum ano com os dois estilos em quantidade suficiente: a comparação ingênua não tem como ser separada da época.'

## 6. Regressão: tudo ao mesmo tempo

Uma regressão linear sobre **log(views)** mede a associação de cada fator **mantendo os outros constantes** (inclusive o ano, que absorve o viés de views acumuladas). Com log na variável resposta, `exp(coef) - 1` se lê como "% a mais de views". Os erros-padrão são robustos (HC3).

In [ ]:
reg = gp.copy()
reg['log_views'] = np.log(reg.views)
reg['log_duration'] = np.log(reg.duration_min)
reg['top_genre'] = reg.genre.where(reg.genre.isin(reg.genre.value_counts().head(8).index), 'Outros')
model = smf.ols('log_views ~ log_duration + is_episode_1 + is_collab + is_challenge + title_has_question'
                ' + title_has_emoji + C(published_year) + C(top_genre)', data=reg).fit(cov_type='HC3')
print(f'R² = {model.rsquared:.2f}  |  n = {int(model.nobs)}')
coefs = pd.DataFrame({'coef': model.params, 'p-valor': model.pvalues})
coefs['efeito %'] = (np.exp(coefs.coef) - 1) * 100
coefs.loc[[i for i in coefs.index if i != 'Intercept' and not i.startswith('C(published_year)')]].round(3)

## 7. Conclusões (escreva com suas palavras)

Sugestão de estrutura, pensando em quem decide a agenda do canal:

1. **O achado mais forte** (com número): …
2. **Recomendação prática:** …
3. **O que os dados NÃO permitem afirmar** (causalidade, viés de views acumuladas, datas estimadas): …
4. **Próximo passo:** que experimento de verdade (A/B de thumbnail ou título, que o YouTube oferece nativamente) testaria a hipótese mais promissora?